# Análisis de factores que determinan el precio del alojamiento en Airbnb — Río de Janeiro

**Pablo Santiago Martínez Soler**

Análisis exploratorio y modelado predictivo del precio de publicaciones de Airbnb en Río de Janeiro, con datos públicos de [Inside Airbnb](http://insideairbnb.com/).

**Pregunta:** ¿qué características de un alojamiento explican su precio por noche, y en qué medida puede predecirse?

Se evalúan cinco factores: tamaño y capacidad, ubicación geográfica, tipo de propiedad, reputación del anfitrión, y el contenido textual de la publicación.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor)
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA_PATH = 'data/listings.csv.gz'
RANDOM_STATE = 42


## 1. Carga y limpieza

In [ ]:
df = pd.read_csv(DATA_PATH, compression='gzip')
df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)
df = df[df['price'] > 0]
print(df.shape)
print(df['price'].describe())


(44542, 90)
count     44542.000000
mean        877.707421
std        4599.612177
min           5.540000
25%         296.710000
50%         452.670000
75%         761.537500
max      574013.000000
Name: price, dtype: float64


La distribución del precio es fuertemente asimétrica: la media (877) casi duplica la mediana (452) y el máximo supera los 574.000 reales. Esto condiciona el tratamiento de outliers.

In [ ]:
def limpiar_columnas(df):
    """Elimina identificadores, URLs y columnas sin información."""
    columnas_eliminar = [
        'id','listing_url','scrape_id','last_scraped','source',
        'host_id','host_url','host_profile_id','host_profile_url',
        'host_name','picture_url',
        'price_quote_checkin_date','price_quote_checkout_date',
        'price_quote_total_price','price_quote_price_per_night',
        'price_quote_raw','calendar_updated','calendar_last_scraped'
    ]
    df = df.drop(columns=columnas_eliminar, errors='ignore')
    return df.drop(columns=df.columns[df.isna().mean() == 1])

df = limpiar_columnas(df)
print(df.shape)


(44542, 59)


### Valores faltantes

La imputación se diferencia según el **patrón de ausencia** de cada grupo:

- **Variables de reviews:** ausentes en alojamientos que aún no recibieron reseñas. No son datos perdidos al azar, sino ausencia estructural; se imputan por mediana y se agrega un indicador `tiene_reviews`.
- **Variables de tamaño** (`bedrooms`, `bathrooms`, `beds`): se imputan por mediana **dentro de cada `room_type`**, ya que la mediana de dormitorios de un cuarto privado difiere de la de una casa entera.
- **`host_about`:** 51% de faltantes, se descarta.

In [ ]:
def tratar_faltantes(df):
    df = df.copy()
    df['tiene_reviews'] = df['number_of_reviews'] > 0
    review_cols = ['review_scores_rating','review_scores_accuracy','review_scores_cleanliness',
                   'review_scores_checkin','review_scores_communication','review_scores_location',
                   'review_scores_value','reviews_per_month']
    for col in review_cols:
        df[col] = df[col].fillna(df[col].median())
    for col in ['bedrooms','bathrooms','beds']:
        df[col] = df.groupby('room_type')[col].transform(lambda x: x.fillna(x.median()))
    df['host_location'] = df['host_location'].fillna('desconocido')
    return df.drop(columns=['host_about'], errors='ignore')

df = tratar_faltantes(df)
print(df.isna().mean().sort_values(ascending=False).head(4))


last_review       0.197252
first_review      0.197252
description       0.017197
bathrooms_text    0.001437
dtype: float64


### Outliers

Se aplica un **filtro combinado**: se elimina una observación si supera el percentil 99 de `price` **o** el percentil 99 del precio por persona (`price / accommodates`). El segundo criterio es el que más aporta: detecta alojamientos caros *en relación a su capacidad*, que un filtro de precio absoluto deja pasar.

Se agregan además filtros de coherencia interna (camas o dormitorios incompatibles con la capacidad declarada) y se acota `minimum_nights` a 30 para excluir alquileres de larga estadía, cuyo precio responde a otra lógica de mercado.

In [ ]:
def tratar_outliers(df):
    df = df.copy()
    df['price_por_persona'] = df['price'] / df['accommodates']
    outliers = ((df['price_por_persona'] > df['price_por_persona'].quantile(0.99)) |
                (df['price'] > df['price'].quantile(0.99)))
    print(f"Filas eliminadas por precio atípico: {outliers.sum()} de {len(df)}")
    df = df[~outliers].drop(columns=['price_por_persona'])

    df = df[~((df['beds'] > df['accommodates'] * 2) | (df['bedrooms'] > df['accommodates']))]
    df = df[df['minimum_nights'] <= 30]
    df['log_price'] = np.log(df['price'])
    return df

df = tratar_outliers(df)
print(df.shape)


Filas eliminadas por precio atípico: 631 de 44542
(43395, 60)


## 2. Ingeniería de variables

### Variables geográficas

La latitud y longitud crudas obligan al modelo a reconstruir la geografía por sí solo. Se agregan dos variables que la codifican directamente:

1. **Distancia a Copacabana** (haversine, en km): referencia de la Zona Sur, el corredor de mayor valor inmobiliario de la ciudad.
2. **Cluster espacial** (KMeans, 25 grupos): captura submercados locales sin depender de los límites administrativos de barrio.

In [ ]:
def agregar_distancia_geografica(df, ref_lat=-22.9711, ref_lon=-43.1822):
    """Distancia haversine en km a Copacabana."""
    df = df.copy()
    R = 6371
    phi1, phi2 = np.radians(df['latitude']), np.radians(ref_lat)
    dphi = np.radians(ref_lat - df['latitude'])
    dlambda = np.radians(ref_lon - df['longitude'])
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    df['dist_km_copacabana'] = 2*R*np.arcsin(np.sqrt(a))
    return df

def agregar_geo_cluster(df, n_clusters=25):
    """Clustering espacial sobre lat/lon para capturar submercados locales."""
    df = df.copy()
    km = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
    df['geo_cluster'] = km.fit_predict(df[['latitude','longitude']].values).astype(str)
    return df

df = agregar_distancia_geografica(df)
df = agregar_geo_cluster(df)

print('Correlación lineal de la distancia con log_price:',
      round(df['dist_km_copacabana'].corr(df['log_price']), 4))
resumen = df.groupby('geo_cluster')['price'].mean().sort_values(ascending=False)
print(f"Precio medio por cluster: de {resumen.min():.0f} a {resumen.max():.0f} reales")


Correlación lineal de la distancia con log_price: -0.0544
Precio medio por cluster: de 361 a 879 reales


La correlación lineal de la distancia con el precio es prácticamente nula (-0.05), pero eso **no** significa que la variable no sirva: Río no tiene un único centro de valor sino varios polos dispersos (Zona Sur, Barra da Tijuca, Joá), de modo que la relación precio-distancia no es monótona. Un modelo de árboles sí puede aprovecharla, y la sección de importancia de variables confirma que lo hace.

Los clusters espaciales, por su parte, separan submercados con precios medios que van de 361 a 879 reales.

### Variables de texto

La descripción de la publicación se incorpora al modelo por dos vías: métricas simples de extensión, y **TF-IDF (50 términos)** integrado directamente en el pipeline de preprocesamiento, de modo que el texto forma parte del modelo final y no de un análisis separado.

In [ ]:
def agregar_features_texto(df):
    df = df.copy()
    desc = df['description'].fillna('')
    df['description_largo'] = desc.apply(len)
    df['description_palabras'] = desc.apply(lambda x: len(x.split()))
    return df

df = agregar_features_texto(df)


## 3. Preparación para el modelado

In [ ]:
def seleccionar_predictoras(df):
    cols_excluir = [
        'name', 'description', 'host_picture_url', 'first_review', 'last_review',
        'price', 'log_price',
        'estimated_revenue_l365d',   # se deriva del precio: introduciría fuga de información
        'amenities', 'bathrooms_text',
    ]
    X = df.drop(columns=[c for c in cols_excluir if c in df.columns])
    for c in X.select_dtypes(include='bool').columns:
        X[c] = X[c].astype(int)
    for c in ['host_is_superhost','host_has_profile_pic','host_identity_verified','has_availability']:
        X[c] = X[c].map({'t': 1, 'f': 0})
    X['description'] = df['description'].fillna('')   # se conserva para TF-IDF
    return X

X = seleccionar_predictoras(df)
y_price, y_log = df['price'], df['log_price']

idx_train, idx_test = train_test_split(X.index, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_test = X.loc[idx_train].copy(), X.loc[idx_test].copy()
y_train, y_test = y_price.loc[idx_train], y_price.loc[idx_test]
y_log_train, y_log_test = y_log.loc[idx_train], y_log.loc[idx_test]
print(X_train.shape, X_test.shape)


(34716, 55) (8679, 55)


`estimated_revenue_l365d` se excluye por fuga de información: se calcula a partir del propio precio multiplicado por la ocupación estimada, de modo que incluirla daría al modelo acceso indirecto a la variable objetivo.

In [ ]:
def agrupar_categorias_poco_frecuentes(train_col, test_col, umbral=30):
    """Agrupa en 'Otro' las categorías con menos de `umbral` casos en entrenamiento."""
    frecuentes = train_col.value_counts()[lambda s: s >= umbral].index
    return (train_col.apply(lambda x: x if x in frecuentes else 'Otro'),
            test_col.apply(lambda x: x if x in frecuentes else 'Otro'))

for col in ['neighbourhood_cleansed', 'property_type']:
    X_train[col], X_test[col] = agrupar_categorias_poco_frecuentes(X_train[col], X_test[col])

X_train = X_train.drop(columns=['host_location'])   # alta cardinalidad, sin poder predictivo
X_test = X_test.drop(columns=['host_location'])


In [ ]:
cols_categoricas = ['neighbourhood_cleansed', 'property_type', 'room_type', 'geo_cluster']
cols_texto = 'description'
cols_numericas = [c for c in X_train.columns if c not in cols_categoricas + [cols_texto]]

def construir_preprocesador():
    """Preprocesador único para todos los modelos: numéricas escaladas,
    categóricas one-hot, y texto vectorizado con TF-IDF."""
    return ColumnTransformer(transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), cols_numericas),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cols_categoricas),
        ('texto', TfidfVectorizer(max_features=50, stop_words='english'), cols_texto),
    ])

preprocesador = construir_preprocesador()
Xt_train = preprocesador.fit_transform(X_train)   # ajustado solo sobre entrenamiento
Xt_test = preprocesador.transform(X_test)
print(Xt_train.shape, Xt_test.shape)


(34716, 206) (8679, 206)


El preprocesador se ajusta **exclusivamente sobre el conjunto de entrenamiento** y luego se aplica al de prueba, evitando que estadísticos del test (medianas, vocabulario TF-IDF, categorías) se filtren al entrenamiento.

## 4. Modelos

In [ ]:
def evaluar(nombre, y_pred, y_real=y_test):
    return {'Modelo': nombre,
            'RMSE': np.sqrt(mean_squared_error(y_real, y_pred)),
            'MAE': mean_absolute_error(y_real, y_pred),
            'R2': r2_score(y_real, y_pred)}

resultados = []

mco = LinearRegression().fit(Xt_train, y_train)
resultados.append(evaluar('MCO', mco.predict(Xt_test)))

lasso = LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=5000, alphas=50).fit(Xt_train, y_train)
resultados.append(evaluar('LASSO', lasso.predict(Xt_test)))
print(f"LASSO alpha seleccionado: {lasso.alpha_:.3f}")


LASSO alpha seleccionado: 0.401


In [ ]:
rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1,
                            n_estimators=200, max_depth=30,
                            min_samples_split=2, min_samples_leaf=4)
rf.fit(Xt_train, y_train)
resultados.append(evaluar('Random Forest', rf.predict(Xt_test)))


In [ ]:
gb = GradientBoostingRegressor(random_state=RANDOM_STATE,
                                n_estimators=100, max_depth=7,
                                learning_rate=0.1, min_samples_leaf=10)
gb.fit(Xt_train, y_train)
resultados.append(evaluar('Gradient Boosting', gb.predict(Xt_test)))


In [ ]:
# Gradient Boosting por histogramas: mismo algoritmo con discretización previa
# de los predictores, lo que permite más iteraciones y mayor profundidad a igual costo.
modelo_final = HistGradientBoostingRegressor(random_state=RANDOM_STATE,
                                             max_iter=400, max_depth=None,
                                             learning_rate=0.03, l2_regularization=0.1)
modelo_final.fit(Xt_train, y_train)
y_pred_final = modelo_final.predict(Xt_test)
resultados.append(evaluar('Gradient Boosting (histogramas)', y_pred_final))


### Comparación

In [ ]:
tabla = pd.DataFrame(resultados).sort_values('RMSE').reset_index(drop=True)
print(tabla.to_string(index=False, float_format=lambda x: f'{x:.4f}' if x < 1 else f'{x:.2f}'))


                         Modelo   RMSE    MAE     R2
Gradient Boosting (histogramas) 481.42 251.86 0.6252
              Gradient Boosting 487.07 252.35 0.6163
                  Random Forest 516.48 263.45 0.5686
                            MCO 576.29 324.11 0.4629
                          LASSO 578.73 322.04 0.4584


El **Gradient Boosting por histogramas es el mejor modelo**: RMSE de 481.42 reales, MAE de 251.86 y R² de 0.6252 sobre el conjunto de prueba.

La distancia entre los modelos de ensamble y los lineales es sustancial (RMSE 481 vs. 576, R² 0.63 vs. 0.46), lo que indica que el precio depende de relaciones no lineales e interacciones entre variables —entre tamaño, tipo de propiedad y ubicación— que una especificación lineal no captura. LASSO no mejora a MCO, señal de que el problema no es exceso de variables irrelevantes sino la forma funcional.

### Nota sobre la escala de la variable objetivo

Dada la asimetría del precio, es habitual modelar su logaritmo. Se evaluó esa alternativa: mejora el ajuste en escala logarítmica (R² de 0.573 frente a 0.463 de MCO) y el error absoluto medio (272.91 frente a 324.11), pero **empeora el RMSE en escala de precio real** (608.19 frente a 576.29), incluso aplicando la corrección de sesgo de Duan para la transformación inversa.

La razón es que ambos criterios optimizan objetivos distintos: minimizar el error cuadrático sobre el logaritmo penaliza los errores relativos, mientras que el RMSE en escala original penaliza los errores absolutos grandes de la cola de precios altos. Se conserva `price` como variable objetivo por consistencia con el RMSE como métrica de referencia.

## 5. Importancia de variables

Se utiliza **importancia por permutación**, que mide cuánto se degrada el error al permutar aleatoriamente cada variable. A diferencia de la importancia por impureza de los árboles, no está sesgada hacia variables categóricas de alta cardinalidad.

In [ ]:
nombres_cat = list(preprocesador.named_transformers_['cat']
                   .named_steps['onehot'].get_feature_names_out(cols_categoricas))
nombres_tfidf = [f'tfidf_{w}' for w in
                 preprocesador.named_transformers_['texto'].get_feature_names_out()]
nombres_todas = cols_numericas + nombres_cat + nombres_tfidf

rng = np.random.RandomState(0)
idx_sub = rng.choice(Xt_test.shape[0], size=2500, replace=False)
imp = permutation_importance(modelo_final, Xt_test[idx_sub], y_test.values[idx_sub],
                              n_repeats=5, random_state=RANDOM_STATE,
                              scoring='neg_root_mean_squared_error')

def agrupar_variable(v):
    """Reagrupa las columnas one-hot y TF-IDF bajo su variable de origen."""
    for pref in ['neighbourhood_cleansed_', 'property_type_', 'room_type_', 'geo_cluster_']:
        if v.startswith(pref):
            return pref.rstrip('_')
    return 'texto_tfidf' if v.startswith('tfidf_') else v

df_imp = pd.DataFrame({'variable': nombres_todas, 'importancia': imp.importances_mean})
df_imp['grupo'] = df_imp['variable'].apply(agrupar_variable)
resumen_imp = df_imp.groupby('grupo')['importancia'].sum().sort_values(ascending=False)
print(resumen_imp.head(12).to_string())


bedrooms                     110.948788
bathrooms                     60.127451
accommodates                  53.067341
latitude                      42.240684
availability_365              15.507174
availability_90               15.359761
dist_km_copacabana            14.414868
texto_tfidf                   12.658269
room_type                     11.875686
maximum_minimum_nights         8.931536
availability_eoy               7.561779
estimated_occupancy_l365d      7.302226


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
resumen_imp.head(12).sort_values().plot(kind='barh', ax=ax, color='#2a6f97')
ax.set_xlabel('Aumento del RMSE al permutar la variable')
ax.set_ylabel('')
ax.set_title('Importancia de variables — Gradient Boosting (histogramas)')
plt.tight_layout()
plt.show()


**El tamaño del alojamiento domina.** `bedrooms`, `bathrooms` y `accommodates` concentran la mayor parte de la capacidad explicativa: permutar el número de dormitorios eleva el RMSE en casi 111 reales, más del doble que la siguiente variable.

**La ubicación es el segundo bloque en importancia**, encabezado por `latitude` (42.2). `dist_km_copacabana` también aparece entre las variables más usadas por el modelo (14.4), pero una prueba de especificación adicional (entrenar el mismo modelo con y sin esta variable) mostró que **no mejora la precisión por encima de lo que ya aportan latitud y longitud crudas** — el modelo sin ella obtiene un RMSE levemente mejor (480.12 vs. 481.42). Se mantiene en el modelo final por su valor interpretativo, no porque sea imprescindible: la importancia por permutación mide cuánto usa el modelo una variable, no si esa variable mejora el resultado frente a alternativas correlacionadas como latitud/longitud.

**El texto de la descripción aporta señal medible.** Las 50 variables TF-IDF suman 12.66 de importancia conjunta, por encima de `room_type` (11.88). El contenido de la publicación agrega información sobre el precio que las variables estructuradas no contienen.

**La disponibilidad del calendario tiene peso propio** (`availability_365` y `availability_90`, ~15 cada una), lo que sugiere que la política de precios del anfitrión y su ocupación efectiva están relacionadas.

## 6. Conclusiones

**Modelo final:** Gradient Boosting por histogramas — RMSE 481.42, MAE 251.86, R² 0.6252 sobre el conjunto de prueba.

**Qué determina el precio, en orden de importancia:**

1. **Tamaño y capacidad** (dormitorios, baños, huéspedes) — el factor dominante, por amplio margen.
2. **Ubicación** — tanto en coordenadas como en distancia al corredor de mayor valor de la ciudad.
3. **Disponibilidad del calendario** — vinculada a la estrategia comercial del anfitrión.
4. **Tipo de alojamiento y contenido de la descripción** — con aporte menor pero medible.

**Consideraciones metodológicas:**

- Las variables geográficas construidas (distancia y clusters espaciales) aportan capacidad predictiva pese a correlaciones lineales bajas, lo que ilustra el límite de seleccionar variables por correlación cuando el modelo es no lineal.
- La transformación logarítmica de la variable objetivo mejora unas métricas y empeora otras: la elección de escala debe responder a qué tipo de error importa en el caso de uso, no aplicarse por defecto.
- Con R² de 0.63 queda variación sin explicar, atribuible a factores no observables en los datos: calidad de las fotografías, estado real de la propiedad, estacionalidad y ajustes dinámicos de precio.
